In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

%pip install -q --upgrade kaggle pyyaml

In [ ]:
import os
import json
import yaml
import shutil
import hashlib
import zipfile
import tempfile
import subprocess

from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

os.environ["KAGGLE_API_TOKEN"] = (
    userdata.get("KAGGLE_API_TOKEN")
)

KAGGLE_USERNAME = "mhassraza"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

assert PROJECT_ROOT.is_dir()

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp", ".bmp"
}

TASKS = {

    "helmet": {
        "model": PROJECT_ROOT /
                 "models/helmet/helmet_best.pt",

        "dataset": PROJECT_ROOT /
                   "datasets/processed/helmet_v3",

        "architecture": "YOLOv8m",

        "names": {
            0: "with_helmet",
            1: "without_helmet"
        },

        "counts": {
            "val": 305,
            "test": 154
        }
    },

    "seatbelt": {
        "model": PROJECT_ROOT /
                 "models/seatbelt/seatbelt_best.pt",

        "dataset": PROJECT_ROOT /
                   "datasets/processed/seatbelt_v1",

        "architecture": "YOLOv8s",

        "names": {
            0: "with_seatbelt",
            1: "without_seatbelt"
        },

        "counts": {
            "val": 787,
            "test": 781
        }
    }
}


def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024**2), b""
        ):
            sha.update(chunk)

    return sha.hexdigest()


def locate_dataset(task, info):

    root = info["dataset"]

    def valid_root(path):

        return (
            (path / "data.yaml").is_file()
            and all(
                (path / split / "images").is_dir()
                and (path / split / "labels").is_dir()
                for split in ["val", "test"]
            )
        )

    if valid_root(root):
        return root

    version = root.name

    candidates = [
        p.parent
        for p in PROJECT_ROOT.rglob("data.yaml")
        if version in p.parts
        and valid_root(p.parent)
    ]

    if len(candidates) == 1:
        return candidates[0]

    # Support archived Phase 4 datasets.
    archives = list(
        PROJECT_ROOT.rglob(f"{version}.zip")
    )

    assert len(archives) == 1, (
        f"Cannot uniquely locate {version}. "
        f"Directories: {candidates}, Archives: {archives}"
    )

    extraction_dir = Path(
        tempfile.mkdtemp(prefix=f"{task}_eval_")
    )

    with zipfile.ZipFile(archives[0]) as z:

        for member in z.infolist():

            target = (
                extraction_dir / member.filename
            ).resolve()

            assert target.is_relative_to(
                extraction_dir.resolve()
            )

        z.extractall(extraction_dir)

    candidates = [
        p.parent
        for p in extraction_dir.rglob("data.yaml")
        if valid_root(p.parent)
    ]

    assert len(candidates) == 1

    return candidates[0]


for task, info in TASKS.items():

    weights = info["model"]
    config = weights.with_suffix(".json")

    assert weights.is_file(), weights
    assert config.is_file(), config

    metadata = json.loads(config.read_text())

    assert sha256_file(weights) == metadata["sha256"]

    info["dataset"] = locate_dataset(task, info)

    original_yaml = yaml.safe_load(
        (info["dataset"] / "data.yaml").read_text()
    )

    names = original_yaml["names"]

    if isinstance(names, list):
        names = dict(enumerate(names))
    else:
        names = {int(k): v for k, v in names.items()}

    assert names == info["names"], (
        f"Unexpected class mapping: {task}"
    )

    print("\n", task.upper())
    print("Model integrity: VERIFIED")
    print("Dataset:", info["dataset"])

    for split, expected in info["counts"].items():

        images = [
            p for p in (
                info["dataset"] / split / "images"
            ).iterdir()
            if p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        labels = list(
            (info["dataset"] / split / "labels")
            .glob("*.txt")
        )

        assert len(images) == len(labels) == expected

        assert {p.stem for p in images} == {
            p.stem for p in labels
        }

        print(
            f"{split}: {len(images)} images, "
            f"{len(labels)} labels"
        )

print("\nALL EVALUATION ASSETS VERIFIED.")

In [ ]:
TRANSFER_DIR = Path(
    tempfile.mkdtemp(
        prefix="trafficguard_final_evaluation_"
    )
)

ARCHIVE_PATH = (
    TRANSFER_DIR / "evaluation_assets.zip"
)

manifest = []
files = []

for task, info in TASKS.items():

    weights = info["model"]
    config = weights.with_suffix(".json")

    files.extend([
        (weights, f"models/{task}/{weights.name}"),
        (config, f"models/{task}/{config.name}")
    ])

    root = info["dataset"]

    for split in ["val", "test"]:

        for folder in ["images", "labels"]:

            source_dir = root / split / folder

            for path in sorted(source_dir.iterdir()):

                if not path.is_file():
                    continue

                allowed = (
                    path.suffix.lower() in IMAGE_EXTENSIONS
                    if folder == "images"
                    else path.suffix == ".txt"
                )

                if allowed:

                    relative = (
                        f"datasets/{task}/{split}/"
                        f"{folder}/{path.name}"
                    )

                    files.append((path, relative))


with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for source, relative in tqdm(
        files,
        desc="Packaging evaluation assets"
    ):

        z.write(source, relative)

        manifest.append({
            "relative_path": relative,
            "size_bytes": source.stat().st_size,
            "sha256": sha256_file(source)
        })


(
    TRANSFER_DIR / "evaluation_manifest.json"
).write_text(
    json.dumps(manifest, indent=4)
)

(
    TRANSFER_DIR / "evaluation_summary.json"
).write_text(
    json.dumps({
        "project": "TrafficGuard AI",
        "phase": "7-8",
        "helmet_val": 305,
        "helmet_test": 154,
        "seatbelt_val": 787,
        "seatbelt_test": 781,
        "archive_sha256": sha256_file(ARCHIVE_PATH)
    }, indent=4)
)

print("Files packaged:", len(manifest))

print(
    "Archive size:",
    round(ARCHIVE_PATH.stat().st_size / 1024**2, 2),
    "MB"
)

print("Evaluation archive:", ARCHIVE_PATH)

In [ ]:
TRANSFER_SESSION = datetime.now().strftime(
    "%Y%m%d-%H%M%S"
)

EVALUATION_DATASET_ID = (
    f"{KAGGLE_USERNAME}/"
    f"trafficguard-final-evaluation-{TRANSFER_SESSION}"
)

metadata = {

    "title": (
        f"TrafficGuard Final Evaluation "
        f"{TRANSFER_SESSION}"
    ),

    "id": EVALUATION_DATASET_ID,

    "licenses": [{"name": "other"}]
}

(
    TRANSFER_DIR / "dataset-metadata.json"
).write_text(
    json.dumps(metadata, indent=4)
)

subprocess.run(
    [
        "kaggle", "datasets", "create",
        "-p", str(TRANSFER_DIR)
    ],
    check=True
)

receipt_dir = (
    PROJECT_ROOT / "results/transfer_receipts"
)

receipt_dir.mkdir(parents=True, exist_ok=True)

(
    receipt_dir /
    f"evaluation_assets_{TRANSFER_SESSION}.json"
).write_text(
    json.dumps({
        "dataset_id": EVALUATION_DATASET_ID,
        "files": len(manifest),
        "archive_sha256": sha256_file(ARCHIVE_PATH)
    }, indent=4)
)

print("\nEVALUATION ASSETS UPLOAD SUBMITTED")

print("Dataset ID:", EVALUATION_DATASET_ID)

print(
    "Dataset URL:",
    f"https://www.kaggle.com/datasets/"
    f"{EVALUATION_DATASET_ID}"
)

In [ ]:
PHASE78_ID = (
    "mhassraza/trafficguard-phase78-results-20261126-081902"
)

assert PHASE78_ID.startswith(
    "mhassraza/trafficguard-phase78-results-"
)

assert callable(
    globals().get("import_kaggle_results")
), (
    "Execute the safe import function from the "
    "Phase 6 transfer notebook first."
)

receipt_78 = import_kaggle_results(

    dataset_id=PHASE78_ID,

    manifest_name="phase78_export_manifest.json",

    project_root=PROJECT_ROOT
)

In [ ]:
import hashlib
import json
import shutil
import tempfile
import zipfile
from datetime import datetime
from pathlib import Path
from tqdm.auto import tqdm


def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def import_kaggle_results(dataset_id: str, manifest_name: str, project_root: Path) -> dict:
    project_root = Path(project_root)
    assert project_root.exists(), f"project_root does not exist: {project_root}"

    tmp_root = Path(tempfile.mkdtemp(prefix="kaggle_import_"))
    try:
        # Download dataset from Kaggle into a temporary folder
        if "kaggle_command" in globals() and callable(globals()["kaggle_command"]):
            kaggle_command("datasets", "download", "-d", dataset_id, "-p", str(tmp_root), "--unzip")
        else:
            import subprocess
            subprocess.run(
                ["kaggle", "datasets", "download", "-d", dataset_id, "-p", str(tmp_root), "--unzip"],
                check=True,
            )

        # Extract any nested .zip archives if present
        for zpath in list(tmp_root.rglob("*.zip")):
            extract_to = zpath.parent / f"_unzipped_{zpath.stem}"
            extract_to.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zpath) as zf:
                for member in zf.infolist():
                    target = (extract_to / member.filename).resolve()
                    assert target.is_relative_to(extract_to.resolve()), f"Unsafe zip member: {member.filename}"
                zf.extractall(extract_to)

        # Locate the export manifest
        manifest_matches = list(tmp_root.rglob(manifest_name))
        assert len(manifest_matches) == 1, (
            f"Expected 1 {manifest_name}, found {len(manifest_matches)}: {manifest_matches}"
        )

        manifest_path = manifest_matches[0]
        source_root = manifest_path.parent
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        assert isinstance(manifest, list) and len(manifest) > 0, "Manifest is empty or invalid."

        # Verify source files and check for conflicts against existing project files
        conflicts = []
        verified_items = []

        for entry in tqdm(manifest, desc=f"Verifying {manifest_name}"):
            rel = Path(entry["relative_path"])
            src = (source_root / rel).resolve()
            dst = (project_root / rel).resolve()

            assert src.is_relative_to(source_root.resolve()), f"Unsafe source path: {rel}"
            assert dst.is_relative_to(project_root.resolve()), f"Unsafe destination path: {rel}"
            assert src.is_file(), f"Missing source file: {src}"
            assert src.stat().st_size == entry["size_bytes"], f"Size mismatch on {rel}"
            assert _sha256_file(src) == entry["sha256"], f"SHA256 mismatch on {rel}"

            if dst.exists():
                if not dst.is_file() or _sha256_file(dst) != entry["sha256"]:
                    conflicts.append(str(dst))

            verified_items.append((src, dst, entry))

        if conflicts:
            raise RuntimeError(
                "Import stopped to protect existing differing files:\n" + "\n".join(conflicts[:25])
            )

        copied = 0
        skipped = 0

        for src, dst, entry in tqdm(verified_items, desc="Importing files into project"):
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                skipped += 1
                continue

            temp_dst = dst.with_name(dst.name + ".importing")
            shutil.copy2(src, temp_dst)
            assert _sha256_file(temp_dst) == entry["sha256"], f"Hash mismatch after copy: {dst}"
            temp_dst.rename(dst)
            copied += 1

        receipt_dir = project_root / "results/transfer_receipts"
        receipt_dir.mkdir(parents=True, exist_ok=True)
        receipt = {
            "dataset_id": dataset_id,
            "manifest_name": manifest_name,
            "imported_at": datetime.now().isoformat(),
            "verified_files": len(manifest),
            "new_files_copied": copied,
            "identical_files_skipped": skipped,
            "files_overwritten": 0,
        }
        receipt_path = receipt_dir / f"{dataset_id.split('/')[-1]}.json"
        receipt_path.write_text(json.dumps(receipt, indent=4), encoding="utf-8")

        print("\nSAFE IMPORT COMPLETED")
        print("Files verified:", len(manifest))
        print("New files copied:", copied)
        print("Identical files skipped:", skipped)
        print("Files overwritten: 0")
        print("Transfer receipt:", receipt_path)
        return receipt
    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)


PHASE78_ID = "mhassraza/trafficguard-phase78-results-20261126-081902"

assert PHASE78_ID.startswith("mhassraza/trafficguard-phase78-results-")

receipt_78 = import_kaggle_results(
    dataset_id=PHASE78_ID,
    manifest_name="phase78_export_manifest.json",
    project_root=PROJECT_ROOT,
)

In [ ]:
import json

PHASE78_DIR = (
    PROJECT_ROOT / "results/final_evaluation"
)

required_files = [

    "tables/confidence_threshold_search.csv",

    "tables/selected_validation_thresholds.csv",

    "tables/independent_test_metrics.csv",

    "tables/independent_test_class_metrics.csv",

    "tables/deployment_test_metrics.csv",

    "tables/deployment_class_metrics.csv",

    "tables/validation_vs_independent_test.csv",

    "tables/inference_speed_benchmark.csv",

    "figures/confidence_optimization.png",

    "figures/validation_vs_independent_test.png",

    "figures/helmet_final_confusion_matrices.png",

    "figures/seatbelt_final_confusion_matrices.png",

    "reports/final_inference_config.json",

    "reports/final_evaluation_summary.json",

    "reports/final_evaluation_report.txt"
]

missing = []

for relative in required_files:

    path = PHASE78_DIR / relative

    if path.is_file():

        print("[FOUND]", relative)

    else:

        print("[MISSING]", relative)

        missing.append(relative)

assert not missing, missing


config = json.loads(
    (
        PHASE78_DIR /
        "reports/final_inference_config.json"
    ).read_text()
)

for task in ["helmet", "seatbelt"]:

    weights = (
        PROJECT_ROOT /
        f"models/{task}/{task}_best.pt"
    )

    assert weights.is_file()

    assert sha256_file(weights) == (
        config["models"][task]["model_sha256"]
    )

    print(
        f"\n{task.upper()} MODEL VERIFIED"
    )

    print(
        "Confidence:",
        config["models"][task]["confidence"]
    )

print("\n" + "="*65)

print("PHASE 7-8 RESULTS VERIFIED")

print("HELMET MODEL: VERIFIED")

print("SEATBELT MODEL: VERIFIED")

print("INDEPENDENT DATASET TESTING: COMPLETED")

print("INFERENCE CONFIGURATION: VERIFIED")

print("GOOGLE DRIVE BACKUP: VERIFIED")

print("NEXT PHASE: COMBINED INFERENCE PIPELINE")

print("="*65)